
# AUFS–DyClust Quickstart (Generic, Susenas-like)

This notebook shows how to run the **AUFS–DyClust (Engine C + auto‑K)** pipeline on a ready-to-cluster mixed‑type dataset.

**Input demo**: `../data/household_clustering_ready.csv` (semicolon `;` separated)

Expected columns include a household identifier (e.g., `HHID`) and mixed features such as `PlantProteinShare`, `FVShare`, `ProcessedShare`, `EnergyBurden`, categorical access variables, etc.


In [1]:
import os, sys
from pathlib import Path

# asumsi notebook berada di notebooks/
ROOT = Path("..").resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

print(f"Project root  : {ROOT}")
print(f"Added to path : {SRC}")


Project root  : /home/jupyter-33220015/dynamic clustering
Added to path : /home/jupyter-33220015/dynamic clustering/src


In [2]:

import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/household_clustering_ready.csv")
df = pd.read_csv(DATA_PATH, sep=";")
df.head(3)


,HHID,PlantProteinShare,FVShare,ProcessedShare,PreparedFoodShare,TobaccoShare,EnergyBurden,NonFoodBurden,FoodShare,EducationShare,...,AccessCommunication,Sanitation,HomeOwnership,CookingFuel,WaterSource,WorkerShare,DependencyRatio,SexHead,AgeHead,N_Working
0,156,0.0,0.181397,0.0,0.106040,0.000000,0.013560,0.539792,0.460208,0.0,...,Tidak,Ada: digunakan RT sendiri,Milik sendiri,Minyak Tanah,Air permukaan,0.0,0.222222,Laki-laki,76,0
1,153,0.0,0.172889,0.0,0.113475,0.000000,0.027035,0.747773,0.252227,0.0,...,Tidak,Ada: digunakan RT sendiri,Milik sendiri,Minyak Tanah,Air permukaan,0.5,0.000000,Laki-laki,60,1
2,157,0.0,0.207990,0.0,0.059633,0.584955,0.014587,0.162329,0.837671,0.0,...,Tidak,Tidak ada fasilitas,Milik sendiri,Kayu bakar,Air permukaan,1.0,0.000000,Laki-laki,58,1



## 1) Minimal run — Engine C + auto‑K

This uses the generic end‑to‑end runner, which internally calls `run_aufs_samba` (Phase A feature selection + Phase B auto‑K) and writes useful artifacts (selected features, cluster assignments, profiles, metrics).


In [3]:

from datetime import datetime
from mixclust.pipeline.generic_end2end import run_generic_end2end
from mixclust.aufs_samba.api import AUFSParams

run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
outdir = Path(f"runs/demo-{run_id}")

# You may tweak params if needed; defaults are reasonable
params = AUFSParams(
    engine_mode="C",
    auto_k=True, c_min=2, c_max=6,

    # ⚡ PERCEPAT REWARD (Langkah #2)
    reward_metric="lsil_fixed_calibrated",   # ← was: lsil / silhouette_gower
    lsil_topk=1,
    per_cluster_proto_if_many=1,
    lsil_proto_sample_cap=200,

    ss_max_n=2000,

    # ⛔ MATIKAN RERANK (Langkah #5)
    enable_rerank=False,
    rerank_mode="none",
    rerank_topk=0,
    shadow_rerank=False,

    # ⚙️ REDUNDANSI (Langkah #1)
    kmsnc_k=5,
    build_redundancy_parallel=True,
    build_redundancy_cache="cache/redundancy_k5.pkl",
    mab_n_jobs=2,   # jangan -1 dulu biar tak OOM

    sa_neighbor_mode="full",
    sa_min_size=2, sa_max_size=None,
    mab_T=12, mab_k=6,
    random_state=42, verbose=True, show_progress=True
)


res = run_generic_end2end(
    df_ready=df,
    outdir=str(outdir),
    id_col="HHID",         # your ID column; set None if not available
    drop_cols=None,        # add non-feature columns here if any
    params=params,
    n_clusters_hint=3,     # only used if auto_k=False
    verbose=True
)

res


[ENGINE] mode=C reward=lsil_fixed_calibrated dynamic_k=True C_eff=auto
[TIME] Redundancy matrix: 683.75s
  (Info) HAC-Gower skipped because N (334229) > 2000.


KeyboardInterrupt: 


## 2) What was produced?

Key files inside `runs/demo-<timestamp>/`:

- `features.csv` — selected feature list

- `cluster_assignments.csv` — (HHID, cluster)

- `profiles.json` / `profiles_table.csv` — aggregated per‑cluster profiles

- `profiles.md` — short narrative

- `metrics_internal.json` — timings, best K, algorithm, etc.

- `config.json` — parameters used


In [ ]:

# Peek selected features and a few assignments
import pandas as pd
features = pd.read_csv(outdir / "features.csv")
assign = pd.read_csv(outdir / "cluster_assignments.csv")
features.head(10), assign.head(10)



## 3) Quick profile visuals (optional)

Simple histograms of a few important indicators per cluster. Feel free to modify.


In [ ]:

import matplotlib.pyplot as plt

assign = pd.read_csv(outdir / "cluster_assignments.csv")
df_join = df.merge(assign, left_on="HHID", right_on="HHID", how="left")

numeric_cols = [
    c for c in ["PlantProteinShare","FVShare","ProcessedShare","PreparedFoodShare",
                "TobaccoShare","EnergyBurden","NonFoodBurden","FoodShare",
                "DDS14_norm","DDS13_noTob_norm","DDS12_noTobPrep_norm"]
    if c in df_join.columns
]

# Plot one by one (no subplots; no explicit colors)
for col in numeric_cols[:6]:  # limit to a handful
    plt.figure()
    for k in sorted(df_join["cluster"].dropna().unique()):
        df_k = df_join[df_join["cluster"] == k]
        df_k[col].plot(kind="hist", bins=40, alpha=0.5, label=f"cluster {int(k)}")
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("count")
    plt.legend()
    plt.show()



## 4) Interpreting segments

Open `profiles_table.csv` and `profiles.md` to see concise summaries. The JSON file `profiles.json` contains detailed statistics (means/medians, lifts, chi‑square/ANOVA p‑values) you can use in the dissertation narrative.
